<a href="https://colab.research.google.com/github/gracenaomi1122/my-first-repo/blob/main/Assignment_Retrieval_Augmented_Generation_(RAG)_Basics_Subjective.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import re
import math
from collections import Counter

hr_document = """Employees who have completed their first six months of probation are eligible to work from home for up to two days per week.

Yes.

All annual leave requests must be submitted at least two weeks in advance through the HR portal for manager approval."""

security_document = """Passwords must contain at least one uppercase letter, one lowercase letter, one number, and one special character.

N/A

Passwords must be changed every 90 days and cannot reuse any of the previous five passwords used on the account."""

sample_question = "What is the work from home policy?"


def chunk_documents(text, source_name):
    """
    Split text into paragraphs on blank lines.
    Skip paragraphs shorter than 50 characters.
    Return a list of dictionaries with text and source.
    """
    chunks = []

    paragraphs = re.split(r"\n\s*\n", text.strip())

    for paragraph in paragraphs:
        paragraph = paragraph.strip()
        if len(paragraph) >= 50:
            chunks.append({
                "text": paragraph,
                "source": source_name
            })

    return chunks


def text_to_vector(text):
    """
    Convert text into a case-insensitive word-frequency vector.
    Only alphanumeric words are extracted.
    """
    words = re.findall(r"\b[a-zA-Z0-9]+\b", text.lower())
    return Counter(words)


def cosine_similarity(vec1, vec2):
    """
    Compute cosine similarity between two word-frequency vectors.
    Returns 0.0 if either vector has zero magnitude.
    """
    if not vec1 or not vec2:
        return 0.0

    # Dot product
    dot_product = sum(vec1[word] * vec2.get(word, 0) for word in vec1)

    # Magnitudes
    magnitude1 = math.sqrt(sum(value ** 2 for value in vec1.values()))
    magnitude2 = math.sqrt(sum(value ** 2 for value in vec2.values()))

    if magnitude1 == 0 or magnitude2 == 0:
        return 0.0

    return dot_product / (magnitude1 * magnitude2)


def retrieve(chunks, question, n_results=3):
    """
    Retrieve the top n_results most similar chunks using cosine similarity.
    Returns a list of (text, source, score) tuples.
    """
    question_vector = text_to_vector(question)
    results = []

    for chunk in chunks:
        chunk_vector = text_to_vector(chunk["text"])
        score = cosine_similarity(question_vector, chunk_vector)

        results.append((
            chunk["text"],
            chunk["source"],
            score
        ))

    results.sort(key=lambda item: item[2], reverse=True)

    return results[:n_results]


if __name__ == "__main__":
    hr_chunks = chunk_documents(hr_document, "HR Policy")
    security_chunks = chunk_documents(security_document, "Security Policy")

    all_chunks = hr_chunks + security_chunks

    print("HR chunks kept:", len(hr_chunks))
    print("Security chunks kept:", len(security_chunks))
    print("Total chunks:", len(all_chunks))
    print()

    print("Top retrieval result:")
    print(retrieve(all_chunks, sample_question, n_results=1))

HR chunks kept: 2
Security chunks kept: 2
Total chunks: 4

Top retrieval result:
[('Employees who have completed their first six months of probation are eligible to work from home for up to two days per week.', 'HR Policy', 0.22677868380553634)]


CHUNKING DOCUMENTS

HR Chunks Kept: 2

HR Chunk 1
----------------------------------------
Source : HR Policy
Text   : Employees who have completed their first six months of probation are eligible to work from home for up to two days per week.

HR Chunk 2
----------------------------------------
Source : HR Policy
Text   : All annual leave requests must be submitted at least two weeks in advance through the HR portal for manager approval.

Security Chunks Kept: 2

Security Chunk 1
----------------------------------------
Source : Security Policy
Text   : Passwords must contain at least one uppercase letter, one lowercase letter, one number, and one special character.

Security Chunk 2
----------------------------------------
Source : Security Policy
Text   : Passwords must be changed every 90 days and cannot reuse any of the previous five passwords used on the account.

Total Chunks: 4

QUESTION
What is the work from home policy?

TOP RETRIEVAL RESULT

Rank : 1
Source : HR Policy
Simil